# Bronze Layer - GCS Geolocation Ingestion

Ingesta de geolocalización desde **Google Cloud Storage** usando la
service account de GCP almacenada en Databricks Secret Scope.

## Restricción arquitectónica
Workspace **Databricks on AWS** → Unity Catalog solo admite external locations
S3: `gs://` **no** se registra en UC. Patrón legacy (doc "Connect to Google Cloud
Storage"): service account key → lectura directa (**bypass de UC**) → materializar
en tabla UC **sobre S3** (gobernada).

## Nota de diagnóstico (serverless)
El runtime trae el conector GCS (`gs://` resuelve), pero **bloquea** aplicar la
conf `fs.gs.*` (`spark.conf.set` rechaza la clave y `sparkContext` no está
disponible en Serverless/Standard access mode). La conf con `{{secrets}}` de la
doc aplica a **cluster clásico** (Spark Config tab). Aquí usamos el equivalente
en serverless: **gcsfs** (SDK Python) — mismo patrón que el notebook Azure ADLS.

## Flujo
1. Instalar gcsfs
2. Montar credenciales desde secret scope
3. Leer el CSV con pandas + gcsfs (bypass UC)
4. Guardar como tabla Delta en `eurobank_bronze.raw.bronze_geo_events`

In [ ]:
%pip install gcsfs --quiet
dbutils.library.restartPython()

In [ ]:
# 2. Credenciales de la service account desde Databricks Secret Scope
SCOPE = "eurobank-secrets"

# Información NO sensible (campos del JSON de la service account)
GCS_SA_INFO = {
    "type": "service_account",
    "project_id": "databricks-508217",
    "client_email": "databricks-gcs-access@databricks-508217.iam.gserviceaccount.com",
    "token_uri": "https://oauth2.googleapis.com/token",
    # Sensibles: vienen del secret scope (redactados en salida de notebook)
    "private_key_id": dbutils.secrets.get(scope=SCOPE, key="gcs-private-key-id"),
    "private_key": dbutils.secrets.get(scope=SCOPE, key="gcs-private-key"),
}

GCS_PATH = "gs://eurobank-geo-oyvixj/location/geolocation_events_batch_001.csv"
print("Credenciales montadas desde secret scope")

In [ ]:
# 3. Lectura con gcsfs (bypass de UC en el origen)
import pandas as pd

pdf = pd.read_csv(GCS_PATH, storage_options={"token": GCS_SA_INFO})
df = spark.createDataFrame(pdf)
print(f"Leidos {df.count()} registros desde GCS")
display(df.limit(5))

In [ ]:
# 4. Materializar en tabla UC (destino gobernado: datos en S3)
table_name = "eurobank_bronze.raw.bronze_geo_events"

df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
print(f"Guardado en {table_name}: {spark.table(table_name).count()} registros")
display(spark.table(table_name).limit(5))